In [1]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from tree.xg_classifier import XGClassifier
from xgboost import XGBClassifier
import shap
from common.helper import load_data_and_scale_pos_weight
from mini_shap.independent_tree_shap import IndependentTreeExplainer
from tree.xg_classifier import XGClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import auc, precision_recall_curve

/home/sakane/Documents/projects/python/mini_xgshap/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
both_bin_path = "../datasets/BRFSS/feat_eng/both_bin.parquet"
both_grp_path = "../datasets/BRFSS/feat_eng/both_grp.parquet"
both_og_path = "../datasets/BRFSS/feat_eng/both_og.parquet"

In [3]:
X_both_og_train, X_both_og_test, y_both_og_train, y_both_og_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_og_path)
X_both_og_train_np, X_both_og_test_np, y_both_og_train_np, y_both_og_test_np = X_both_og_train.to_numpy().astype(np.float32), X_both_og_test.to_numpy().astype(np.float32), y_both_og_train.to_numpy().astype(np.float32), y_both_og_test.to_numpy().astype(np.float32)

X_both_og_train_fold, X_both_og_train_val, y_both_og_train_fold, y_both_og_train_val = train_test_split(
    X_both_og_train,
    y_both_og_train,
    test_size=0.2,
    random_state=42,
    stratify=y_both_og_train
)

In [4]:
X_both_grp_train, X_both_grp_test, y_both_grp_train, y_both_grp_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_grp_path)
X_both_grp_train_np, X_both_grp_test_np, y_both_grp_train_np, y_both_grp_test_np = X_both_grp_train.to_numpy().astype(np.float32), X_both_grp_test.to_numpy().astype(np.float32), y_both_grp_train.to_numpy().astype(np.float32), y_both_grp_test.to_numpy().astype(np.float32)

X_both_grp_train_fold, X_both_grp_train_val, y_both_grp_train_fold, y_both_grp_train_val = train_test_split(
    X_both_grp_train,
    y_both_grp_train,
    test_size=0.2,
    random_state=42,
    stratify=y_both_grp_train
)

In [5]:
X_both_bin_train, X_both_bin_test, y_both_bin_train, y_both_bin_test, scale_pos_weight = load_data_and_scale_pos_weight(cast_string=True, dataset_path=both_bin_path)
X_both_bin_train_np, X_both_bin_test_np, y_both_bin_train_np, y_both_bin_test_np = X_both_bin_train.to_numpy().astype(np.float32), X_both_bin_test.to_numpy().astype(np.float32), y_both_bin_train.to_numpy().astype(np.float32), y_both_bin_test.to_numpy().astype(np.float32)

X_both_bin_train_fold, X_both_bin_train_val, y_both_bin_train_fold, y_both_bin_train_val = train_test_split(
    X_both_bin_train,
    y_both_bin_train,
    test_size=0.2,
    random_state=42,
    stratify=y_both_bin_train
)

In [6]:
model_both_og = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    reg_lambda=1.0,
    eval_metric="aucpr",
    early_stopping_rounds=10,
    n_jobs=os.cpu_count() - 2,
    random_state=42,
    enable_categorical=True,
    max_cat_to_onehot=1,
    objective='binary:logistic'
)

model_both_og.fit(X_both_og_train_fold, y_both_og_train_fold, eval_set=[(X_both_og_train_val, y_both_og_train_val)], verbose=False)
model_both_og.save_model("../datasets/BRFSS/feat_eng/models/both_og_model.ubj")

In [7]:
model_both_bin = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    reg_lambda=1.0,
    eval_metric="aucpr",
    early_stopping_rounds=10,
    n_jobs=os.cpu_count() - 2,
    random_state=42,
    enable_categorical=True,
    max_cat_to_onehot=1,
    objective='binary:logistic'
)

model_both_bin.fit(X_both_bin_train_fold, y_both_bin_train_fold, eval_set=[(X_both_bin_train_val, y_both_bin_train_val)], verbose=False)
model_both_bin.save_model("../datasets/BRFSS/feat_eng/models/both_bin_model.ubj")

In [8]:
model_both_grp = XGBClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3,
    reg_lambda=1.0,
    eval_metric="aucpr",
    early_stopping_rounds=10,
    n_jobs=os.cpu_count() - 2,
    random_state=42,
    enable_categorical=True,
    max_cat_to_onehot=1,
    objective='binary:logistic'
)

model_both_grp.fit(X_both_grp_train_fold, y_both_grp_train_fold, eval_set=[(X_both_grp_train_val, y_both_grp_train_val)], verbose=False)
model_both_grp.save_model("../datasets/BRFSS/feat_eng/models/both_grp_model.ubj")